In [13]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from typing import TypedDict
from dotenv import load_dotenv

In [14]:
load_dotenv()

True

In [15]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    max_new_tokens=256,
    temperature=0.7,
)

In [16]:
model = ChatHuggingFace(llm=llm)

In [17]:
# create a state

class LLMState(TypedDict):

    question: str
    answer: str

In [18]:

def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state

In [19]:
# create our graph

graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [20]:
# execute

intial_state = {'question': 'How far is moon from the earth?'}

final_state = workflow.invoke(intial_state)

print(final_state['answer'])

The average distance from the Earth to the Moon is about 238,855 miles (384,400 kilometers). This distance is constantly changing due to the elliptical shape of the Moon's orbit around the Earth. At its closest point (called perigee), the Moon is about 225,622 miles (363,300 kilometers) away from Earth. At its farthest point (called apogee), the Moon is about 252,088 miles (406,700 kilometers) away from Earth.


In [21]:

model.invoke('How far is moon from the earth?').content

"The average distance from the Earth to the Moon is about 384,400 kilometers (238,900 miles). This distance is constantly changing due to the elliptical shape of the Moon's orbit around the Earth. At its closest point (called perigee), the distance is about 384,400 km, and at its farthest point (apogee), it is about 405,500 km.\n\nHere are some interesting facts about the distance from the Earth to the Moon:\n\n1. **Perigee and Apogee:** The Moon's orbit is elliptical, which means that its distance from the Earth varies over the course of a month. The closest point (perigee) is about 384,400 km, and the farthest point (apogee) is about 405,500 km.\n2. **Average distance:** The average distance from the Earth to the Moon is about 384,400 km. This is the distance used when calculating the time it takes for a spacecraft to travel to the Moon (about 3.7 days).\n3. **Lunar orbit:** The Moon takes about 27.3 days to complete one orbit around the Earth, which is called a sidereal month. This 